# Set up Selenium

*INFO 4617 Web Data Science · Week 8 · before Wednesday's lab*

Chapter 8's code controls a real Chrome window with **Selenium**, a Python package that Anaconda doesn't include. Until you install it, chapter 8's browser code stops at its first line. This notebook installs Selenium. Then it lets Selenium download what it needs to control Chrome, and opens Chrome once to show that everything works.

**How to run it.** Click the first gray box of code below (a *cell*) and press **Shift+Enter**. That runs the cell and moves to the next one. You don't need to read or change the code: run each cell, and read what it prints.

- **While a cell runs**, `[*]` shows at its left. Some cells take a few minutes while they download.
- **When it finishes**, a number takes the star's place. Read the last line the cell printed:
  - **OK**: go on to the next cell.
  - **FIX**: do what it says, then run that cell again. Don't go on until it says OK.

**Stuck?** The setup page this notebook came from, [github.com/cuinfoscience/INFO4617-Fall2026/tree/main/handouts/week-08](https://github.com/cuinfoscience/INFO4617-Fall2026/tree/main/handouts/week-08), lists what to do when something goes wrong.

## 1 · Install Selenium

A **package** is code that someone else wrote, which you add to Python. Anaconda came with many packages, such as `pandas` and `requests`, but not Selenium. The next cell installs it with **pip**, Python's installer. `%pip` at the start of a line runs pip for the Python behind this notebook, which is Anaconda's, so you don't need a terminal. pip downloads about 12 MB from the Python Package Index, [pypi.org](https://pypi.org/project/selenium/).

Run the cell. It takes about a minute and prints a few dozen lines. Look near the end:

- `Successfully installed`, followed by a list that includes `selenium-4.`: Selenium is installed.
- `Requirement already satisfied: selenium`: you had it already, which is fine.
- `Note: you may need to restart the kernel to use updated packages.`: you can ignore it this time.

pip may also print warnings about other packages. The cell after it checks whether the install worked.

In [ ]:
# Install Selenium. "--progress-bar off" keeps pip's output short in a notebook.
%pip install --progress-bar off selenium

In [ ]:
try:
    import selenium
except ModuleNotFoundError:
    print("FIX: Selenium isn't installed. Read the last lines of the cell above.")
    print('If they say "Access is denied" or "Permission denied", add a cell (the + button in the toolbar),')
    print("type  %pip install --user selenium  in it, run it, and then run this cell again.")
    print("Otherwise, check your internet connection and run the cell above again.")
else:
    if tuple(int(n) for n in selenium.__version__.split(".")[:2]) < (4, 20):
        print("FIX: Selenium", selenium.__version__, "is too old for chapter 8. Add a cell, type")
        print("  %pip install --upgrade selenium  in it, and run it. Then choose Kernel > Restart Kernel...")
        print("and run this notebook again from the top.")
    else:
        print("OK: Selenium", selenium.__version__, "is installed.")

## 2 · Settings for Selenium Manager

Selenium came with **Selenium Manager**, a small program. When your code starts Chrome, Selenium Manager finds your Chrome and gets a **driver** for it. A driver is a small program, made for your exact version of Chrome, through which Python controls the browser. You never download a driver yourself.

Selenium Manager needs no setup, but the two settings in the next cell make it more reliable and more private. The settings last until you close this notebook. Chapter 8's notebook makes them again, in its step 1.

In [ ]:
import importlib.util
import os
import sys
from pathlib import Path

os.environ["SE_SKIP_DRIVER_IN_PATH"] = "true"  # ignore old drivers that other software left on your computer
os.environ["SE_AVOID_STATS"] = "true"          # don't send Selenium's makers usage statistics
# os.environ["SE_PROXY"] = "http://proxy.example.edu:3128"  # only on a network that needs a proxy

# Only matters in the course's webdata environment, whose Selenium keeps Selenium Manager outside the
# package (`conda activate webdata` would tell Selenium where). With Anaconda as it came, it does nothing.
manager = Path(sys.prefix) / ("Scripts/selenium-manager.exe" if os.name == "nt" else "bin/selenium-manager")
if "SE_MANAGER_PATH" not in os.environ and manager.is_file():
    os.environ["SE_MANAGER_PATH"] = str(manager)

if importlib.util.find_spec("selenium") is None:
    print("FIX: Selenium isn't installed yet. Go back to step 1.")
else:
    print("OK: settings made. Selenium Manager:", os.environ.get("SE_MANAGER_PATH", "inside the selenium package"))

## 3 · Let Selenium Manager get the driver

This cell runs Selenium Manager by itself, before any browser opens, and shows what it decides. The first time, it downloads the driver, about 10 MB. If you have no Chrome, it also downloads **Chrome for Testing**, a copy of Chrome made for code to control, about 190 MB. On a slow connection, that takes several minutes.

Its report has a pink background. That doesn't mean something failed: Jupyter colors everything sent to Python's error stream, and Selenium writes its report there. Read it from the top:

- `Detected browser: chrome` and a version number: you have Chrome, so only the driver downloads.
- `chrome not found in the system`: you have no Chrome, so Chrome for Testing downloads too.
- `Driver path:` and `Browser path:`: the files it chose. What it downloads goes into a folder named `.cache/selenium` in your home folder, so later runs download nothing.

In [ ]:
import logging
from logging.handlers import BufferingHandler

from selenium.webdriver.common.selenium_manager import SeleniumManager

logging.basicConfig(level=logging.WARNING, format="%(levelname)s %(message)s")
log = logging.getLogger("selenium.webdriver.common.selenium_manager")
kept = BufferingHandler(capacity=1000)  # keeps Selenium Manager's warnings for step 4
kept.setLevel(logging.WARNING)

log.addHandler(kept)
log.setLevel(logging.DEBUG)  # show each decision Selenium Manager makes
try:
    paths = SeleniumManager().binary_paths(["--browser", "chrome"])
    print("OK: Selenium Manager found a browser and a driver. Step 4 checks them.")
except Exception as error:
    paths = None
    print("FIX: Selenium Manager stopped:", error)
    if "Selenium Manager binary" in str(error):
        print("Selenium couldn't find Selenium Manager: run step 2, then this cell again.")
    else:
        print('If it says "error sending request", or mentions a connection or a timeout, your network')
        print("blocked the download. Try another network, such as home Wi-Fi, and run this cell again.")
finally:
    log.setLevel(logging.WARNING)  # from here on, show only its warnings
    log.removeHandler(kept)
manager_warnings = [record.getMessage() for record in kept.buffer]

## 4 · Check what Selenium Manager found

Start the browser only when this cell says OK. OK means both files are there, and the driver is one that Selenium Manager chose for your Chrome, not an old one from somewhere else on your computer. The cell also prints the folder where Selenium Manager keeps its downloads.

In [ ]:
import os
from pathlib import Path

cache = Path(os.environ.get("SE_CACHE_PATH", Path.home() / ".cache" / "selenium"))
problems = []
if "paths" not in globals():
    problems.append("step 3 hasn't finished. Run it, do what it says, then run this cell again.")
elif paths is None:
    problems.append("step 3 didn't finish. Do what it says, then run step 3 and this cell again.")
else:
    driver_path = Path(paths["driver_path"])
    browser_path = Path(paths["browser_path"])
    print("Driver: ", driver_path, "(found)" if driver_path.is_file() else "(MISSING)")
    print("Browser:", browser_path, "(found)" if browser_path.is_file() else "(MISSING)")
    print("Selenium Manager's folder:", cache)
    if not (driver_path.is_file() and browser_path.is_file()):
        problems.append("a file is missing. Run step 3 again, then this cell.")
    if cache not in driver_path.parents:
        problems.append("that driver isn't Selenium Manager's. Delete it, or run step 2, then steps 3 and 4 again.")
    for warning in manager_warnings:
        print("Selenium Manager warned:", warning)

if problems:
    for problem in problems:
        print("FIX:", problem)
elif manager_warnings:
    print("OK, with the warning above. Go on to step 5; if it fails, the warning is the likely cause.")
else:
    print("OK: a browser, and a driver that matches it.")

## 5 · Open Chrome, visit a page, close it

Chapter 8 starts the browser with one line, `webdriver.Chrome()`. Run the next cell: a Chrome window opens. Don't click in it or close it. The cell visits a page, prints the page's title, and closes the window itself. Chrome for Testing shows a bar saying that it's only for automated testing, which is expected.

In [ ]:
from selenium import webdriver

driver = None
try:
    driver = webdriver.Chrome()
    print("Browser:", driver.capabilities["browserVersion"])
    print("Driver: ", driver.capabilities["chrome"]["chromedriverVersion"].split()[0])
    driver.get("https://quotes.toscrape.com/")
    print("Page title:", driver.title)
    print("OK: Selenium started Chrome, visited a page, and read its title. You're ready for chapter 8.")
except Exception as error:
    message = str(error)
    print(message)
    if "Unable to obtain" in message:
        print("FIX: Selenium couldn't get a driver. Run steps 2 to 4, then this cell again.")
    elif "only supports Chrome version" in message:
        print("FIX: an old driver was used. Run steps 2 to 4 (step 2 tells Selenium to skip it), then this cell again.")
    elif "Chrome instance exited" in message:
        print("FIX: Chrome closed as it started. On Ubuntu 24.04 or later, install Google Chrome from")
        print("https://www.google.com/chrome/ and run steps 3 to 5 again. Otherwise, bring this to the lab.")
    else:
        print("FIX: bring everything this cell printed to Wednesday's lab.")
finally:
    if driver is not None:
        driver.quit()  # always close the browser: each one is a whole Chrome, using memory

## You're ready for chapter 8

If step 5 said OK, you're done, and you won't need to do this again:

- **Selenium stays installed** in Anaconda's Python.
- **The downloads stay too.** Selenium Manager keeps the driver, and Chrome for Testing if it downloaded, in its folder.
- **Only step 2's settings go away** when you close this notebook. Chapter 8's notebook makes them itself.

**Next: chapter 8's notebook.**

1. Download it the way you downloaded this one, with GitHub's **Download raw file** button, from [its page on GitHub](https://github.com/cuinfoscience/Web-Data-Science-Book/blob/main/notebooks/ch-08-dynamic-pages.ipynb).
2. Open it from Jupyter's list of files.
3. Run its cells in order. Its Selenium cells start under "Before the First Browser", and "Starting the Browser" opens Chrome. To be sure before Wednesday, run it that far.

**Why five steps?** `webdriver.Chrome()` does three jobs in one line: it reads the settings, runs Selenium Manager, and starts the browser. When that line fails, its error rarely says which job went wrong. Steps 2, 3, and 5 do those jobs one at a time, and step 4 checks Selenium Manager's work before the browser starts. Chapter 8's "Before the First Browser" explains each job, and its "When Selenium Manager Fails" lists more causes and fixes.

**Stuck?** See "When something goes wrong" on [the setup page](https://github.com/cuinfoscience/INFO4617-Fall2026/tree/main/handouts/week-08), or bring everything the failing cell printed, from its first line to its last, to Wednesday's lab.